<h1>Fortress Effect</h1>
This notebook builds an end-to-end predictive pipeline using 25 seasons of Premier League match data (9,380 matches / 18,760 team-match rows) to isolate the true venue effect using Logistic Regression while strictly preventing data leakage.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression

<h3>Load and inspect</h3>
This section loads historical Premier League match data spanning 25 seasons (2000/01 to 2024/25) and imports key library dependencies for statistical analysis and machine learning.

In [2]:
df=pd.read_csv("epl_final.csv")
df.head()

,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,...,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,...,14,4,6,6,13,12,1,2,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,...,10,5,7,7,19,14,1,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,...,3,9,8,4,15,21,5,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,...,4,6,5,8,11,13,1,1,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,...,8,6,6,4,21,20,1,3,0,0


In [3]:
df.isna().sum()

Season               0
MatchDate            0
HomeTeam             0
AwayTeam             0
FullTimeHomeGoals    0
FullTimeAwayGoals    0
FullTimeResult       0
HalfTimeHomeGoals    0
HalfTimeAwayGoals    0
HalfTimeResult       0
HomeShots            0
AwayShots            0
HomeShotsOnTarget    0
AwayShotsOnTarget    0
HomeCorners          0
AwayCorners          0
HomeFouls            0
AwayFouls            0
HomeYellowCards      0
AwayYellowCards      0
HomeRedCards         0
AwayRedCards         0
dtype: int64

In [4]:
df.dtypes

Season                 str
MatchDate              str
HomeTeam               str
AwayTeam               str
FullTimeHomeGoals    int64
FullTimeAwayGoals    int64
FullTimeResult         str
HalfTimeHomeGoals    int64
HalfTimeAwayGoals    int64
HalfTimeResult         str
HomeShots            int64
AwayShots            int64
HomeShotsOnTarget    int64
AwayShotsOnTarget    int64
HomeCorners          int64
AwayCorners          int64
HomeFouls            int64
AwayFouls            int64
HomeYellowCards      int64
AwayYellowCards      int64
HomeRedCards         int64
AwayRedCards         int64
dtype: object

In [5]:
df.groupby('Season').size()

Season
2000/01    380
2001/02    380
2002/03    380
2003/04    335
2004/05    335
2005/06    380
2006/07    380
2007/08    380
2008/09    380
2009/10    380
2010/11    380
2011/12    380
2012/13    380
2013/14    380
2014/15    380
2015/16    380
2016/17    380
2017/18    380
2018/19    380
2019/20    380
2020/21    380
2021/22    380
2022/23    380
2023/24    380
2024/25    350
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
sorted(df['HomeTeam'].unique())

['Arsenal',
 'Aston Villa',
 'Birmingham',
 'Blackburn',
 'Blackpool',
 'Bolton',
 'Bournemouth',
 'Bradford',
 'Brentford',
 'Brighton',
 'Burnley',
 'Cardiff',
 'Charlton',
 'Chelsea',
 'Coventry',
 'Crystal Palace',
 'Derby',
 'Everton',
 'Fulham',
 'Huddersfield',
 'Hull',
 'Ipswich',
 'Leeds',
 'Leicester',
 'Liverpool',
 'Luton',
 'Man City',
 'Man United',
 'Middlesbrough',
 'Newcastle',
 'Norwich',
 "Nott'm Forest",
 'Portsmouth',
 'QPR',
 'Reading',
 'Sheffield United',
 'Southampton',
 'Stoke',
 'Sunderland',
 'Swansea',
 'Tottenham',
 'Watford',
 'West Brom',
 'West Ham',
 'Wigan',
 'Wolves']

In [8]:
df['FullTimeResult'].value_counts(normalize=True)

FullTimeResult
H    0.458316
A    0.295096
D    0.246588
Name: proportion, dtype: float64

<h3>Clean</h3>

In [9]:
df['MatchDate']=pd.to_datetime(df['MatchDate'],dayfirst=True)

C:\Users\vishw\AppData\Local\Temp\ipykernel_12728\1039008030.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['MatchDate']=pd.to_datetime(df['MatchDate'],dayfirst=True)


In [10]:
df.dtypes

Season                          str
MatchDate            datetime64[us]
HomeTeam                        str
AwayTeam                        str
FullTimeHomeGoals             int64
FullTimeAwayGoals             int64
FullTimeResult                  str
HalfTimeHomeGoals             int64
HalfTimeAwayGoals             int64
HalfTimeResult                  str
HomeShots                     int64
AwayShots                     int64
HomeShotsOnTarget             int64
AwayShotsOnTarget             int64
HomeCorners                   int64
AwayCorners                   int64
HomeFouls                     int64
AwayFouls                     int64
HomeYellowCards               int64
AwayYellowCards               int64
HomeRedCards                  int64
AwayRedCards                  int64
dtype: object

In [11]:
df=df.sort_values('MatchDate').reset_index(drop=True)

In [12]:
counts=df.groupby('Season').size()
counts[counts<380]

Season
2003/04    335
2004/05    335
2024/25    350
dtype: int64

In [13]:
s=df[df['Season']=='2003/04']
s['HomeTeam'].value_counts()


HomeTeam
Birmingham       18
Man United       18
Leicester        18
Southampton      18
Everton          18
Middlesbrough    18
Fulham           17
Leeds            17
Newcastle        17
Tottenham        17
Arsenal          16
Blackburn        16
Portsmouth       16
Charlton         16
Wolves           16
Man City         16
Chelsea          16
Bolton           16
Aston Villa      16
Liverpool        15
Name: count, dtype: int64

In [14]:
(df['HomeShots']==0).sum()

np.int64(2)

In [15]:
(df['HomeFouls'] == 0).sum()

np.int64(3)

In [16]:
df.columns


Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards'],
      dtype='str')

In [17]:
df=df.drop(columns=[ 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards'])

<h3>Reshape</h3>

In [18]:
home_df=df[['Season','MatchDate','HomeTeam','AwayTeam','FullTimeHomeGoals','FullTimeAwayGoals']].copy()

In [19]:
home_df.head()

,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals
0,2000/01,2000-08-19,Charlton,Man City,4,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3
3,2000/01,2000-08-19,Derby,Southampton,2,2
4,2000/01,2000-08-19,Leeds,Everton,2,0


In [20]:
home_df=home_df.rename(columns={
    'HomeTeam': 'team',
    'AwayTeam': 'opponent',
    'FullTimeHomeGoals': 'goals_for',
    'FullTimeAwayGoals': 'goals_against'})

In [21]:
home_df['is_home']=1
home_df['won']=(df['FullTimeResult'] == 'H').astype(int)

In [22]:
home_df.head()

,Season,MatchDate,team,opponent,goals_for,goals_against,is_home,won
0,2000/01,2000-08-19,Charlton,Man City,4,0,1,1
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,1,1
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,1,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,1,1


In [23]:
away_df=df[['Season','MatchDate','HomeTeam','AwayTeam','FullTimeHomeGoals','FullTimeAwayGoals']].copy()

away_df=away_df.rename(columns={
    'HomeTeam': 'opponent',
    'AwayTeam': 'team',
    'FullTimeHomeGoals': 'goals_against',
    'FullTimeAwayGoals': 'goals_for'})


In [24]:
away_df['is_home']=0
away_df['won']=(df['FullTimeResult'] == 'A').astype(int)
away_df.head()

,Season,MatchDate,opponent,team,goals_against,goals_for,is_home,won
0,2000/01,2000-08-19,Charlton,Man City,4,0,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,0,1
3,2000/01,2000-08-19,Derby,Southampton,2,2,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,0,0


In [25]:
cols = ['Season', 'MatchDate', 'team', 'opponent', 'is_home', 'goals_for', 'goals_against', 'won']
away_df = away_df[cols]
home_df = home_df[cols]

In [26]:
new_df = pd.concat([home_df, away_df])
new_df = new_df.sort_values('MatchDate').reset_index(drop=True)
new_df.head(20)

,Season,MatchDate,team,opponent,is_home,goals_for,goals_against,won
0,2000/01,2000-08-19,Charlton,Man City,1,4,0,1
1,2000/01,2000-08-19,Southampton,Derby,0,2,2,0
2,2000/01,2000-08-19,Everton,Leeds,0,0,2,0
3,2000/01,2000-08-19,Aston Villa,Leicester,0,0,0,0
4,2000/01,2000-08-19,Bradford,Liverpool,0,0,1,0
5,2000/01,2000-08-19,Arsenal,Sunderland,0,0,1,0
6,2000/01,2000-08-19,Ipswich,Tottenham,0,1,3,0
7,2000/01,2000-08-19,Man City,Charlton,0,0,4,0
8,2000/01,2000-08-19,Middlesbrough,Coventry,0,3,1,1
9,2000/01,2000-08-19,West Ham,Chelsea,0,2,4,0


<h3>Sanity Checks</h3>

In [27]:
len(new_df)

18760

In [28]:
new_df['is_home'].mean()

np.float64(0.5)

In [29]:
new_df.groupby('is_home')['won'].mean()

is_home
0    0.295096
1    0.458316
Name: won, dtype: float64

<h3>Features</h3>

In [30]:

win=new_df['goals_for']>new_df['goals_against']
draw=new_df['goals_for']==new_df['goals_against']
new_df['points']=np.select([win,draw],[3,1],default=0)
new_df.head()

,Season,MatchDate,team,opponent,is_home,goals_for,goals_against,won,points
0,2000/01,2000-08-19,Charlton,Man City,1,4,0,1,3
1,2000/01,2000-08-19,Southampton,Derby,0,2,2,0,1
2,2000/01,2000-08-19,Everton,Leeds,0,0,2,0,0
3,2000/01,2000-08-19,Aston Villa,Leicester,0,0,0,0,1
4,2000/01,2000-08-19,Bradford,Liverpool,0,0,1,0,0


In [31]:
new_df = new_df.sort_values(['team','MatchDate']).reset_index(drop=True)
new_df.head(10)

,Season,MatchDate,team,opponent,is_home,goals_for,goals_against,won,points
0,2000/01,2000-08-19,Arsenal,Sunderland,0,0,1,0,0
1,2000/01,2000-08-21,Arsenal,Liverpool,1,2,0,1,3
2,2000/01,2000-08-26,Arsenal,Charlton,1,5,3,1,3
3,2000/01,2000-09-06,Arsenal,Chelsea,0,2,2,0,1
4,2000/01,2000-09-09,Arsenal,Bradford,0,1,1,0,1
5,2000/01,2000-09-16,Arsenal,Coventry,1,2,1,1,3
6,2000/01,2000-09-23,Arsenal,Ipswich,0,1,1,0,1
7,2000/01,2000-10-01,Arsenal,Man United,1,1,0,1,3
8,2000/01,2000-10-14,Arsenal,Aston Villa,1,1,0,1,3
9,2000/01,2000-10-21,Arsenal,West Ham,0,2,1,1,3


In [32]:
new_df['form_5'] = (new_df.groupby('team')['points'].transform(lambda s:s.shift(1).rolling(5,min_periods=1).mean()))
new_df.head(20)

,Season,MatchDate,team,opponent,is_home,goals_for,goals_against,won,points,form_5
0,2000/01,2000-08-19,Arsenal,Sunderland,0,0,1,0,0,NaN
1,2000/01,2000-08-21,Arsenal,Liverpool,1,2,0,1,3,0.00
2,2000/01,2000-08-26,Arsenal,Charlton,1,5,3,1,3,1.50
3,2000/01,2000-09-06,Arsenal,Chelsea,0,2,2,0,1,2.00
4,2000/01,2000-09-09,Arsenal,Bradford,0,1,1,0,1,1.75
5,2000/01,2000-09-16,Arsenal,Coventry,1,2,1,1,3,1.60
6,2000/01,2000-09-23,Arsenal,Ipswich,0,1,1,0,1,2.20
7,2000/01,2000-10-01,Arsenal,Man United,1,1,0,1,3,1.80
8,2000/01,2000-10-14,Arsenal,Aston Villa,1,1,0,1,3,1.80
9,2000/01,2000-10-21,Arsenal,West Ham,0,2,1,1,3,2.20


In [33]:
opp=new_df[['MatchDate','team','form_5']].rename(columns={'team':'opponent','form_5':'opp_form_5'})

In [34]:
new_df = new_df.merge(opp, on=['MatchDate','opponent'], how='left')
new_df.head(20)

,Season,MatchDate,team,opponent,is_home,goals_for,goals_against,won,points,form_5,opp_form_5
0,2000/01,2000-08-19,Arsenal,Sunderland,0,0,1,0,0,NaN,NaN
1,2000/01,2000-08-21,Arsenal,Liverpool,1,2,0,1,3,0.00,3.000000
2,2000/01,2000-08-26,Arsenal,Charlton,1,5,3,1,3,1.50,1.500000
3,2000/01,2000-09-06,Arsenal,Chelsea,0,2,2,0,1,2.00,1.333333
4,2000/01,2000-09-09,Arsenal,Bradford,0,1,1,0,1,1.75,1.000000
5,2000/01,2000-09-16,Arsenal,Coventry,1,2,1,1,3,1.60,1.400000
6,2000/01,2000-09-23,Arsenal,Ipswich,0,1,1,0,1,2.20,1.400000
7,2000/01,2000-10-01,Arsenal,Man United,1,1,0,1,3,1.80,2.200000
8,2000/01,2000-10-14,Arsenal,Aston Villa,1,1,0,1,3,1.80,2.000000
9,2000/01,2000-10-21,Arsenal,West Ham,0,2,1,1,3,2.20,1.200000


In [35]:
new_df['gd']=new_df['goals_for']-new_df['goals_against']


In [36]:
new_df=new_df.sort_values(['team','MatchDate']).reset_index(drop=True)
new_df['GD10']=(new_df.groupby('team')['gd'].transform(lambda s:s.shift(1).rolling(10,min_periods=3).mean()))
new_df.head(10)

,Season,MatchDate,team,opponent,is_home,goals_for,goals_against,won,points,form_5,opp_form_5,gd,GD10
0,2000/01,2000-08-19,Arsenal,Sunderland,0,0,1,0,0,NaN,NaN,-1,NaN
1,2000/01,2000-08-21,Arsenal,Liverpool,1,2,0,1,3,0.00,3.000000,2,NaN
2,2000/01,2000-08-26,Arsenal,Charlton,1,5,3,1,3,1.50,1.500000,2,NaN
3,2000/01,2000-09-06,Arsenal,Chelsea,0,2,2,0,1,2.00,1.333333,0,1.000000
4,2000/01,2000-09-09,Arsenal,Bradford,0,1,1,0,1,1.75,1.000000,0,0.750000
5,2000/01,2000-09-16,Arsenal,Coventry,1,2,1,1,3,1.60,1.400000,1,0.600000
6,2000/01,2000-09-23,Arsenal,Ipswich,0,1,1,0,1,2.20,1.400000,0,0.666667
7,2000/01,2000-10-01,Arsenal,Man United,1,1,0,1,3,1.80,2.200000,1,0.571429
8,2000/01,2000-10-14,Arsenal,Aston Villa,1,1,0,1,3,1.80,2.000000,1,0.625000
9,2000/01,2000-10-21,Arsenal,West Ham,0,2,1,1,3,2.20,1.200000,1,0.666667


In [37]:
opgd=new_df[['MatchDate','team','GD10']].rename(columns={'team':'opponent','GD10':'opp_GD10'})
new_df=new_df.merge(opgd,on=['MatchDate','opponent'],how='left')
new_df.head(10)

,Season,MatchDate,team,opponent,is_home,goals_for,goals_against,won,points,form_5,opp_form_5,gd,GD10,opp_GD10
0,2000/01,2000-08-19,Arsenal,Sunderland,0,0,1,0,0,NaN,NaN,-1,NaN,NaN
1,2000/01,2000-08-21,Arsenal,Liverpool,1,2,0,1,3,0.00,3.000000,2,NaN,NaN
2,2000/01,2000-08-26,Arsenal,Charlton,1,5,3,1,3,1.50,1.500000,2,NaN,NaN
3,2000/01,2000-09-06,Arsenal,Chelsea,0,2,2,0,1,2.00,1.333333,0,1.000000,0.000000
4,2000/01,2000-09-09,Arsenal,Bradford,0,1,1,0,1,1.75,1.000000,0,0.750000,-1.250000
5,2000/01,2000-09-16,Arsenal,Coventry,1,2,1,1,3,1.60,1.400000,1,0.600000,-0.400000
6,2000/01,2000-09-23,Arsenal,Ipswich,0,1,1,0,1,2.20,1.400000,0,0.666667,-0.333333
7,2000/01,2000-10-01,Arsenal,Man United,1,1,0,1,3,1.80,2.200000,1,0.571429,1.857143
8,2000/01,2000-10-14,Arsenal,Aston Villa,1,1,0,1,3,1.80,2.000000,1,0.625000,0.571429
9,2000/01,2000-10-21,Arsenal,West Ham,0,2,1,1,3,2.20,1.200000,1,0.666667,-0.111111


In [38]:
len(new_df)

18760

<h3>Split by Time</h3>

In [39]:
features=['is_home','form_5','opp_form_5','GD10','opp_GD10']
model_df=new_df.dropna(subset=features+['won']).copy()

train=model_df[model_df['Season']<'2019/20']
test=model_df[model_df['Season']>='2019/20']

x_train, y_train=train[features], train['won']
x_test, y_test=test[features], test['won']

In [40]:
len(train)

14062

In [41]:
len(test)

4482

In [42]:
y_train.mean()

np.float64(0.37419997155454415)

<h3>Fit the Model</h3>

In [43]:
model = LogisticRegression(max_iter=1000)

In [44]:
model.fit(x_train,y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [45]:
coefs=pd.Series(model.coef_[0], index=features)
print(coefs)
print('intercept :',model.intercept_[0])

is_home       0.906239
form_5       -0.006491
opp_form_5   -0.034965
GD10          0.634025
opp_GD10     -0.564464
dtype: float64
intercept : -0.984010643473344


In [46]:
print(model.coef_)
print(model.coef_[0])
print(model.intercept_)
print(model.intercept_[0])

[[ 0.9062388  -0.00649055 -0.03496468  0.6340253  -0.56446441]]
[ 0.9062388  -0.00649055 -0.03496468  0.6340253  -0.56446441]
[-0.98401064]
-0.984010643473344


<h3>Evaluate</h3>

In [47]:
pred = model.predict(x_test)
proba = model.predict_proba(x_test)[:,1]

In [48]:
print("model accuracy:", accuracy_score(y_test, pred))
print("model log loss:", log_loss(y_test, proba))
print("baseline accuracy:", 1 - y_test.mean())
print("baseline log loss:", log_loss(y_test, [y_train.mean()] * len(y_test)))


model accuracy: 0.6639892904953146
model log loss: 0.6120916878291294
baseline accuracy: 0.6153502900490853
baseline log loss: 0.6665268764126407


In [49]:
print(len(train), len(test), y_test.mean())

14062 4482 0.3846497099509148


<h3>Counterfactual Flip</h3>

In [50]:
# 1. Force all matches to be Home
X_home = x_test.copy()
X_home['is_home'] = 1

# 2. Force all matches to be Away
X_away = x_test.copy()
X_away['is_home'] = 0

# 3. Predict win probabilities for both scenarios
p_home = model.predict_proba(X_home)[:, 1]
p_away = model.predict_proba(X_away)[:, 1]

# 4. Print safety checks and results
print("X_home values:", X_home['is_home'].unique())
print("X_away values:", X_away['is_home'].unique())
print("Mean Home Win Proba:", p_home.mean())
print("Mean Away Win Proba:", p_away.mean())
print("Home Advantage (diff):", (p_home - p_away).mean())


X_home values: [1]
X_away values: [0]
Mean Home Win Proba: 0.4701317851721537
Mean Away Win Proba: 0.2839349827286533
Home Advantage (diff): 0.1861968024435004


<h3>Team by Team</h3>

In [51]:
seasons=new_df.groupby('team')['Season'].nunique()
seasons.head(50)

team
Arsenal             25
Aston Villa         22
Birmingham           7
Blackburn           11
Blackpool            1
Bolton              11
Bournemouth          8
Bradford             1
Brentford            4
Brighton             8
Burnley              9
Cardiff              2
Charlton             7
Chelsea             25
Coventry             1
Crystal Palace      13
Derby                3
Everton             25
Fulham              18
Huddersfield         2
Hull                 5
Ipswich              3
Leeds                7
Leicester           13
Liverpool           25
Luton                1
Man City            24
Man United          25
Middlesbrough       10
Newcastle           23
Norwich              7
Nott'm Forest        3
Portsmouth           7
QPR                  3
Reading              3
Sheffield United     4
Southampton         17
Stoke               10
Sunderland          14
Swansea              7
Tottenham           25
Watford              7
West Brom           13
West H

In [52]:
eligible = seasons[seasons > 10].index
print(len(eligible), list(eligible))

19 ['Arsenal', 'Aston Villa', 'Blackburn', 'Bolton', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Leicester', 'Liverpool', 'Man City', 'Man United', 'Newcastle', 'Southampton', 'Sunderland', 'Tottenham', 'West Brom', 'West Ham', 'Wolves']


In [53]:
filtered=new_df[new_df['team'].isin(eligible)]

In [54]:
rates=filtered.groupby(['team','is_home'])['won'].mean().unstack()
rates['gap'] = rates[1]-rates[0]
rates.sort_values('gap',ascending=False)

is_home,0,1,gap
team,,,
Fulham,0.169643,0.423881,0.254238
Tottenham,0.351812,0.586354,0.234542
Newcastle,0.257541,0.474419,0.216878
Everton,0.267666,0.473461,0.205795
Arsenal,0.469083,0.670940,0.201857
Man City,0.460000,0.651111,0.191111
Liverpool,0.473461,0.657388,0.183927
Man United,0.498934,0.678038,0.179104
West Ham,0.245192,0.407674,0.162482


In [55]:
extra = filtered.groupby('team').agg(
    matches=('won', 'size'),
    seasons=('Season', 'nunique'))

In [56]:
rates = rates.join(extra)
rates['home_games'] = rates['matches'] / 2
rates.sort_values('gap', ascending=False)

,0,1,gap,matches,seasons,home_games
team,,,,,,
Fulham,0.169643,0.423881,0.254238,671,18,335.5
Tottenham,0.351812,0.586354,0.234542,938,25,469.0
Newcastle,0.257541,0.474419,0.216878,861,23,430.5
Everton,0.267666,0.473461,0.205795,938,25,469.0
Arsenal,0.469083,0.670940,0.201857,937,25,468.5
Man City,0.460000,0.651111,0.191111,900,24,450.0
Liverpool,0.473461,0.657388,0.183927,938,25,469.0
Man United,0.498934,0.678038,0.179104,938,25,469.0
West Ham,0.245192,0.407674,0.162482,833,22,416.5


In [57]:
# quick summary of everything above

home_adv = (p_home - p_away).mean()
acc = accuracy_score(y_test, pred)
base_acc = 1 - y_test.mean()
ll = log_loss(y_test, proba)
base_ll = log_loss(y_test, [y_train.mean()] * len(y_test))

print("Fortress Effect - final results")
print("-" * 40)
print(f"Matches: {len(new_df) // 2} (2000/01 to 2024/25)")#as one match is one's home and opponet's away
print(f"Train rows: {len(train)}, test rows: {len(test)}")
print()
print(f"Win chance at home: {p_home.mean() * 100:.1f}%")
print(f"Win chance away:    {p_away.mean() * 100:.1f}%")
print(f"So home advantage is about {home_adv * 100:.1f} points")
print(f"(playing at home multiplies the odds of winning by ~{np.exp(coefs['is_home']):.1f}x)")
print()
print(f"Model accuracy {acc * 100:.1f}% vs {base_acc * 100:.1f}% for always guessing 'no win'")
print(f"Log loss {ll:.3f} vs {base_ll:.3f} for the lazy model (lower is better)")
print()
print(f"Biggest home boost: {rates['gap'].idxmax()} ({rates['gap'].max() * 100:.1f} pts)")
print(f"Smallest: {rates['gap'].idxmin()} ({rates['gap'].min() * 100:.1f} pts)")
print()
print("Not perfect: draws count as not a win, the test years include empty-stadium 2020/21,")
print("and the team gaps ignore how strong the opponents were.")

Fortress Effect - final results
----------------------------------------
Matches: 9380 (2000/01 to 2024/25)
Train rows: 14062, test rows: 4482

Win chance at home: 47.0%


Win chance away:    28.4%
So home advantage is about 18.6 points
(playing at home multiplies the odds of winning by ~2.5x)

Model accuracy 66.4% vs 61.5% for always guessing 'no win'
Log loss 0.612 vs 0.667 for the lazy model (lower is better)

Biggest home boost: Fulham (25.4 pts)
Smallest: Crystal Palace (6.0 pts)

Not perfect: draws count as not a win, the test years include empty-stadium 2020/21,
and the team gaps ignore how strong the opponents were.


In [58]:
top_team = rates['gap'].idxmax()
print(f"Biggest home boost: {top_team} ({rates['gap'].max() * 100:.1f} pts)")
print(f"But {top_team} only played {rates.loc[top_team, 'seasons']} seasons "
      f"({rates.loc[top_team, 'matches']} matches), so don't read too much into it.")
print()

big_six = ['Man United','Arsenal', 'Chelsea', 'Liverpool','Tottenham']
compare = rates.loc[[top_team] + big_six, ['gap', 'seasons', 'matches']].copy()
compare['gap'] = (compare['gap'] * 100).round(1)
print(compare)

Biggest home boost: Fulham (25.4 pts)
But Fulham only played 18 seasons (671 matches), so don't read too much into it.

             gap  seasons  matches
team                              
Fulham      25.4       18      671
Man United  17.9       25      938
Arsenal     20.2       25      937
Chelsea     13.8       25      937
Liverpool   18.4       25      938
Tottenham   23.5       25      938
